## El siguiente ejemplo muestra algunas ideas para automatizar transformaciones y evitar la fuga de datos.

Ajustado el 21/05/2026, sepan entender cualquier error!

In [69]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

#cache
from joblib import Memory


In [70]:
#Imputador con columnas, estrategia: composición
class Plantilla(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass        
    def fit(self, X, y=None):
        return self    
    def get_feature_names_out(self):
        return None
    def transform(self, X):
        return None

In [71]:
df = pd.DataFrame({"col1":[33, 99, 55, 99, 999, 55,  "16", 999,  33, 999, 44, 55, 88, 55],
                   "col2":[88, 99, 55, 99, 999, 55,  16, 999,  91, 999, 44, 77, 88, 33],
                   "col3":[-0.92, -1.46,  1.92 , np.nan , -0.38, 1.47,  2.81, -1.79, np.nan,  1.22, -0.92, 1.47,  2.81, -1.55],
                   "cat":  ["B", "B", "M", "B", "B", "M", "M", "B", "M", "M", "B", "M", "M", "B"],
                   "target":[0,   0,   1,   0,   0,   1,   1,   0,   1,   1,   0,   1,   1,   1]})



In [72]:
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=3, test_size=0.3)

In [73]:
class TypeSelector(BaseEstimator, TransformerMixin):
    def __init__(self, dtype_map):
        self.dtype_map = dtype_map # Ejemplo: {'col1': 'float', 'col2': 'int'}
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        X = X.copy()
        return X.astype(self.dtype_map)


class IntegerImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='mean', number=999):
        super().__init__()
        self.strategy = strategy
        self.number = number
        self.imputer = None

    def _replace_number(self, X):
        if isinstance(X, pd.DataFrame):
            return X.replace(self.number, np.nan)
        if isinstance(X, np.ndarray):
            return np.where(X == self.number, np.nan, X)
        raise TypeError("Input type not supported. Expected pandas DataFrame or numpy array.")

    def fit(self, X, y=None):
        Xc = self._replace_number(X)
        self.imputer = SimpleImputer(strategy=self.strategy)
        self.imputer.set_output(transform="pandas")
        self.imputer.fit(Xc)
        return self

    def get_feature_names_out(self, input_features=None):
        return self.imputer.get_feature_names_out(input_features)

    def transform(self, X):
        Xc = self._replace_number(X)
        Xt = self.imputer.transform(Xc)
        return Xt.astype(int)

In [74]:
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=3, test_size=0.3)
plcol1 = Pipeline(steps=[
    ("type_selector", TypeSelector(dtype_map={"col1": "int"})),
    ("IntegerImputer", IntegerImputer(strategy='mean', number=999)),
])

ct = ColumnTransformer(transformers=[
    ("col1Transformer", plcol1, ["col1"]),
    ("col2Transformer", IntegerImputer(strategy='mean', number=999), ["col2"]),
    ("col3Transformer", SimpleImputer(strategy='mean'), ["col3"]),
    ("OneHot", OneHotEncoder(sparse_output=False, drop='first', dtype=int), ["cat"])
],)
ct.fit_transform(X_train)

array([[16.        , 16.        ,  2.81      ,  1.        ],
       [55.        , 55.        ,  1.47      ,  1.        ],
       [33.        , 88.        , -0.92      ,  0.        ],
       [55.        , 77.        ,  1.47      ,  1.        ],
       [88.        , 88.        ,  2.81      ,  1.        ],
       [99.        , 99.        ,  1.13428571,  0.        ],
       [52.        , 69.        ,  1.22      ,  1.        ],
       [33.        , 91.        ,  1.13428571,  1.        ],
       [44.        , 44.        , -0.92      ,  0.        ]])

In [75]:
pl = Pipeline(steps=[
    ("ct", ct),
    
])

La columna col3, ¿Sera mejor media o mediana? ¿Sera mejor un árbol de 2,3 o 4 de profundidad?

In [76]:
#hagamos la clasificacion
pl.steps.append(("clf", DecisionTreeClassifier(max_depth=3)))


In [77]:
pl

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('col1Transformer',
                                                  Pipeline(steps=[('type_selector',
                                                                   TypeSelector(dtype_map={'col1': 'int'})),
                                                                  ('IntegerImputer',
                                                                   IntegerImputer())]),
                                                  ['col1']),
                                                 ('col2Transformer',
                                                  IntegerImputer(), ['col2']),
                                                 ('col3Transformer',
                                                  SimpleImputer(), ['col3']),
                                                 ('OneHot',
                                                  OneHotEncoder(drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['cat'])])),
                ('clf', DecisionTreeClassifier(max_depth=3))])

In [78]:
pl[:-1].fit_transform(X_train)

array([[16.        , 16.        ,  2.81      ,  1.        ],
       [55.        , 55.        ,  1.47      ,  1.        ],
       [33.        , 88.        , -0.92      ,  0.        ],
       [55.        , 77.        ,  1.47      ,  1.        ],
       [88.        , 88.        ,  2.81      ,  1.        ],
       [99.        , 99.        ,  1.13428571,  0.        ],
       [52.        , 69.        ,  1.22      ,  1.        ],
       [33.        , 91.        ,  1.13428571,  1.        ],
       [44.        , 44.        , -0.92      ,  0.        ]])

In [79]:
pl.fit(X_train, y_train)
y_pred = pl.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       1.00      0.50      0.67         2

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5



Podriamos probar...GridSearchCV...

1.Hacerlo de forma correcta, el pipeline dentro del gridsearch.  
2.Para mejorar rendimiento se puede hacer uso de joblib para cachear los resultados de las transformaciones, evitando recalcularlas en cada ejecución.  
3.La regla para parametrizar el gridsearch es <nombre_del_paso>__<nombre_del_parametro>, si tengo elementos internos debo seguir la misma política, una mamuschka en definitiva.

In [80]:
pl

Pipeline(steps=[('ct',
                 ColumnTransformer(transformers=[('col1Transformer',
                                                  Pipeline(steps=[('type_selector',
                                                                   TypeSelector(dtype_map={'col1': 'int'})),
                                                                  ('IntegerImputer',
                                                                   IntegerImputer())]),
                                                  ['col1']),
                                                 ('col2Transformer',
                                                  IntegerImputer(), ['col2']),
                                                 ('col3Transformer',
                                                  SimpleImputer(), ['col3']),
                                                 ('OneHot',
                                                  OneHotEncoder(drop='first',
                                                                dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['cat'])])),
                ('clf', DecisionTreeClassifier(max_depth=3))])

In [81]:
for step in pl.steps:
    print(step[0])  

ct
clf


In [82]:
cachedir = './Customtransformer_cache_dir'
memory = Memory(location=cachedir, verbose=0)

param_grid = {
    "ct__col3Transformer__strategy": ["mean", "median"],
    "clf__max_depth":[1,2,3]
}


gs = GridSearchCV(pl, param_grid=param_grid, cv=2)
gs.fit(X_train, y_train)
y_pred = gs.predict(X_test)
print(classification_report(y_test, y_pred))

del memory

              precision    recall  f1-score   support

           0       0.75      1.00      0.86         3
           1       1.00      0.50      0.67         2

    accuracy                           0.80         5
   macro avg       0.88      0.75      0.76         5
weighted avg       0.85      0.80      0.78         5

